In [2]:
cd /work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io

/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io


In [3]:
datadir = '/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/struct_treesmk2/'


In [4]:
#run pdbfixer on all structs
from Bio import PDB
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import glob
import sys
import tqdm
import pandas as pd
import multiprocessing as mp
#argument parser for the script 
from concurrent.futures import TimeoutError
from pebble import ProcessPool, ProcessExpired
import os
import glob

In [5]:
#create a dataframe with all the pdb files
maindir = '/work/FAC/FBM/DBC/cdessim2/default/dmoi/projects/aars/aarsonline.github.io/'
c1 = glob.glob(maindir + 'class1/*/*/structures/*.pdb')
c2 = glob.glob(maindir + 'class2/*/*/structures/*.pdb')


domains = glob.glob(maindir + '*/*/data/domains/*/structures/*.pdb')

all_pdbs = c1 + c2 + domains
#remove duplicates
all_pdbs = list(set(all_pdbs))

pdbs = { i:{ 'class': pdb.split('/')[11], 'res': pdb.split('/')[12] , 'path': pdb } for i,pdb in enumerate(all_pdbs) }
print(f'Found {len(pdbs)} pdb files')

Found 3371 pdb files


In [6]:

pdb_df = pd.DataFrame.from_dict(pdbs, orient='index')
pdb_df['pdb_id'] = pdb_df['path'].apply(lambda x: os.path.basename(x).split('.')[0])
pdb_df['domain'] = pdb_df['path'].apply(lambda x: x.split('/')[15] if 'domains' in x else None)
print(pdb_df.head())

    class   res                                               path  \
0  class1  leu1  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
1  class2   his  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
2  class1   arg  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
3  class2  gly1  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   
4  class2  asp1  /work/FAC/FBM/DBC/cdessim2/default/dmoi/projec...   

                                              pdb_id            domain  
0  LeuRS-B_AF_Bact_Bifidobacterium_longum_subsp_l...              None  
1                 HisRS_AF_Euk_Homo_sapiens_gene3035              None  
2                    ArgRS_PDB_Euk_S_cerevisiae_1BS2  Catalytic_domain  
3  GlyRS-A_AF_Arch_Candidatus_Woesearchaeota_arch...  Catalytic_domain  
4  AspRS_AF_Bact_Porphyromonas_gingivalis_ATCC_33...              None  


In [7]:
#print the number of pdb files per class
print(pdb_df['class'].value_counts())

class
class1    1750
class2    1621
Name: count, dtype: int64


In [8]:
print( pdb_df['domain'].value_counts() )

domain
Catalytic_domain                 548
Protozyme                        548
Anticodon_binding_domain_1a       99
CP2                               99
Anticodon_binding_domain_HGPT     97
                                ... 
Lysine-rich                        3
Beta_chain                         3
Ergosterol_binding_domain          1
ProRS                              1
GluRS                              1
Name: count, Length: 64, dtype: int64


In [9]:
#drop all except Protozyme and Catalytic domains
keep = ['Protozyme', 'Catalytic_domain' , 'Anticodon_binding_domain_CRIMVL' , 'Anticodon_binding_domain']
pdb_df = pdb_df[pdb_df['domain'].isin(keep)]
print(f'Keeping {len(pdb_df)} pdb files after filtering for domains {keep}')

Keeping 1175 pdb files after filtering for domains ['Protozyme', 'Catalytic_domain', 'Anticodon_binding_domain_CRIMVL', 'Anticodon_binding_domain']


In [10]:
#print the number of pdb files per class / residue
print(pdb_df.groupby(['class', 'res']).size())

class   res 
class1  arg     72
        cys     92
        gln     30
        glu1    38
        glu2    10
        glu3    14
        ile     48
        leu1    32
        leu2    22
        lys     18
        met     50
        trp     50
        tyr     50
        val     46
class2  ala     40
        asn     44
        asp1    28
        asp2    30
        gly1    16
        gly2    27
        gly3    12
        his     46
        lys     44
        phe1    20
        phe2    24
        phe3    20
        phe4    18
        phe5    14
        pro1    32
        pro2    40
        pyl     18
        sep     18
        ser1    54
        ser2    10
        thr     48
dtype: int64


In [11]:
structs_all = glob.glob(datadir+'*/structures/*.pdb')
print( len(structs_all))

0


In [12]:
#how many fams are there? group by class and residue and domain
print(pdb_df.groupby(['class', 'res', 'domain']).size())
#are there any empty classes?
print(pdb_df[pdb_df['class'].isnull()])

class   res   domain                         
class1  arg   Anticodon_binding_domain_CRIMVL    24
              Catalytic_domain                   24
              Protozyme                          24
        cys   Anticodon_binding_domain           23
              Anticodon_binding_domain_CRIMVL    23
                                                 ..
class2  ser1  Protozyme                          27
        ser2  Catalytic_domain                    5
              Protozyme                           5
        thr   Catalytic_domain                   24
              Protozyme                          24
Length: 75, dtype: int64
Empty DataFrame
Columns: [class, res, path, pdb_id, domain]
Index: []


In [13]:
#make a directory for the results
if not os.path.exists(datadir):
	os.makedirs(datadir )

clear = True
#make a folder for each class, residue and domain
pdb_df['struct_dir'] = pdb_df.apply(lambda x: os.path.join(datadir, x['class'], x['res'], x['domain']), axis=1)
for struct_dir in pdb_df['struct_dir'].unique():
	if not os.path.exists(struct_dir):
		os.makedirs(struct_dir)
		if not os.path.exists(os.path.join(struct_dir, 'structs')):
			os.makedirs(os.path.join(struct_dir, 'structs'))
		if clear == True:
			#remove all pdb files in the structs directory
			for pdb_file in glob.glob(os.path.join(struct_dir, 'structs', '*.pdb')):
				os.remove(pdb_file)
		
		if not os.path.exists(os.path.join(struct_dir, 'pdbfixer_in')):
			os.makedirs(os.path.join(struct_dir, 'pdbfixer_in'))

	#create a dummy identifiers.txt file
	with open(os.path.join(struct_dir, 'identifiers.txt'), 'w') as f:
		f.write('This is a dummy file to indicate that this directory has been processed by pdbfixer.\n')
		f.write('You can remove this file if you want to reprocess the directory.\n')


In [14]:
#print the tree structure of the directories
def print_tree(path, level=0):
	if os.path.isdir(path):
		print(' ' * level + os.path.basename(path) + '/')
		for item in os.listdir(path):
			print_tree(os.path.join(path, item), level + 2)
	else:
		print(' ' * level + os.path.basename(path))

print_tree(datadir)

/
  logs/
    IX_mad_root_structML/
    BM_foldseek2distmat/
    BM_foldseek_allvall_0/
    BM_ML_mad_root_postML/
    BM_postprocess/
    FX_treescore/
    BM_mad_root_seq/
    BM_ML_iqtree_template/
    BM_dl_ids_sequences/
    FX_aln2distmat/
    BM_mad_root_iq/
    BM_ML_foldseek_createdb/
    BM_ML_calcfident_distributions/
    BM_ML_mafft_seq/
    BM_ML_treescore/
    BM_ML_iqtreex/
    BM_fasttree/
    BM_ML_mafft_textaln/
    IX_iqtree3di/
    BM_ML_cross_alns/
    BM_foldseek_allvall_1/
    BM_ML_scrape_alns/
    BM_calc_tax_score_iq/
    BM_quicktree/
    IX_treescore/
    BM_plddt/
    FX_mad_root_post/
    BM_iqtree/
    BM_calc_tax_score_seq/
    IX_mad_root_postML/
    FX_postprocess/
    BM_calc_tax_score/
    BM_ML_mad_root_structML/
    BM_muscle/
    BM_mad_root_struct/
    BM_dl_ids_structs/
    FX_mad_root_struct/
    FX_quicktree/
  class1/
    cys/
      Catalytic_domain/
        3di_xaln_tree.nwk
        fident_1_raw_struct_tree.PP.nwk.rooted
        alntmscore_1

KeyboardInterrupt: 

In [15]:
#copy all pdb files to the pdbfixer_in directory
for i, row in tqdm.tqdm(pdb_df.iterrows() , total=len(pdb_df), desc='Copying PDB files to pdbfixer_in'):
	src = row['path']
	dest = os.path.join(row['struct_dir'], 'pdbfixer_in', os.path.basename(src))
	if not os.path.exists(dest):
		os.makedirs(os.path.dirname(dest), exist_ok=True)
		os.system(f'cp {src} {dest}')

Copying PDB files to pdbfixer_in: 100%|██████████| 1175/1175 [00:00<00:00, 16252.11it/s]


In [16]:
def prepchain( pdbfile , chain=None, savepath=None ,unid='',  verbose = False):
	assert savepath is not None
	try:
		#parse the pdb file
		parser = PDB.PDBParser()
		structure = parser.get_structure(unid, pdbfile)
		if chain:
			chainID = chain
		else:
			chainID = list(structure[0].get_chains())[0].get_id()
		chain_A = structure[0][chainID]
	except:
		print("error")
		return None
	if chain_A:
		io = PDB.PDBIO()
		io.set_structure(chain_A)
		io.save(savepath)
		fixer = PDBFixer(filename=savepath)
		fixer.findNonstandardResidues()
		fixer.replaceNonstandardResidues()
		fixer.removeHeterogens(True)
		fixer.findMissingResidues()
		fixer.findMissingAtoms()
		fixer.addMissingAtoms()
		#fixer.addMissingHydrogens(7.0)
		PDBFile.writeFile(fixer.topology, fixer.positions, open(savepath, 'w'))
		return None


In [17]:
structs = glob.glob(datadir + '*/*/*/pdbfixer_in/*.pdb')
print(f'Found {len(structs)} pdb files to process')

Found 1175 pdb files to process


In [18]:
fix_structs = True
if fix_structs:
	#remove all pdb files that have already been processed
	structs = [ pdb for pdb in structs if not os.path.exists(pdb.replace('pdbfixer_in', 'structs')) or not os.path.exists(pdb.replace('pdbfixer_in', 'identifiers.txt')) ]
	print(f'Found {len(structs)} pdb files to process after filtering')
	with ProcessPool() as pool:
		futures = [ pool.schedule( prepchain,  ( pdb , None , pdb.replace('pdbfixer_in', 'structs' ) 
										  , '' , False ) , timeout = 240) for pdb in structs  ]
		for future in tqdm.tqdm(futures, total=len(structs)):
			try:
				results = future.result()
			except TimeoutError as error:
				print("unstable_function took longer than %d seconds" % error.args[1])
			except ProcessExpired as error:
				print("%s. Exit code: %d" % (error, error.exitcode))
			except Exception as error:
				print("unstable_function raised %s" % error)
				print(error.traceback)  # Python's traceback of remote process
		pool.close()
		pool.join()


Found 1175 pdb files to process after filtering


  0%|          | 0/1175 [00:00<?, ?it/s]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 1383.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 1478.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 1551.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 1779.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuil

error


/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 2692.
  warnings.warn(
 14%|█▍        | 169/1175 [03:06<42:40,  2.55s/it]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 865.
  warnings.warn(
 15%|█▌        | 180/1175 [03:07<18:26,  1.11s/it]/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 2986.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/StructureBuilder.py:89: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 3112.
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/en

error


/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 388
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 1577
  warnings.warn(
/work/FAC/FBM/DBC/cdessim2/default/dmoi/miniconda3/envs/torch/lib/python3.12/site-packages/Bio/PDB/PDBParser.py:388: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 301
  warnings.warn(
100%|██████████| 1175/1175 [14:45<00:00,  1.33it/s]


In [19]:
#swap name to number for the pdb files and save a mapping file
#rename the pdb files in the structs directory
def rename_pdb_files(struct_dir):
	structs = glob.glob(os.path.join(struct_dir, 'structs', '*.pdb'))
	mapping = {}
	for i,pdb in enumerate(structs):
		pdb_id = os.path.basename(pdb).split('.')[0]
		new_name = f'{i}.pdb'
		new_path = os.path.join(struct_dir, 'structs', new_name)
		os.rename(pdb, new_path)
		#save the mapping
		mapping[pdb_id] = new_name
	#transform the mapping to a dataframe
	mapping_df = pd.DataFrame.from_dict(mapping, orient='index', columns=['new_name'])
	mapping_df.index.name = 'pdb_id'
	#save the mapping to a file
	mapping_df.to_csv(os.path.join(struct_dir, 'mapping.csv'))
#apply the renaming function to all struct directories
for struct_dir in tqdm.tqdm(pdb_df['struct_dir'].unique(), desc='Renaming PDB files'):
	rename_pdb_files(struct_dir)

Renaming PDB files: 100%|██████████| 75/75 [00:04<00:00, 16.08it/s]
